In [1]:
import torch
import torchaudio

print(torch.__version__)
print(torchaudio.__version__)

2.8.0+cu128
2.8.0+cu128


In [2]:
from fairseq2 import gang
gang._thread_local.current_gangs = []

In [3]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
pipeline = ASRInferencePipeline(model_card= 'omniASR_LLM_300M')

Output()

In [4]:
target_langs = ['eng_Latn', 'swh_Latn']

In [5]:
import subprocess
from pathlib import Path

def encode_to_wav(audio):
    encoded_audio = subprocess.run(
        ['ffmpeg', '-i', audio, '-f', 'wav', 'pipe:1'],
        check = True,
        capture_output= True
    )
    return encoded_audio.stdout #stdout is the actual file

audio = Path('sw-test-speech-1.m4a')
encoded_audio = encode_to_wav(audio)
type(encoded_audio) #type(encoded_audio)

bytes

In [6]:
import io, soundfile as sf
#obtained bytes go to a memory like object

audio = io.BytesIO(encoded_audio)
audio.seek(0)

waveform, sr = sf.read(audio)
waveform = torch.from_numpy(waveform).float()

In [7]:
print(f'Sample rate: {sr} Hz')

Sample rate: 48000 Hz


In [8]:
import tempfile

with tempfile.NamedTemporaryFile(suffix='.wav', delete= False) as tmp:
    sf.write(tmp.name, waveform.numpy(), sr)
    transcript = pipeline.transcribe([tmp.name], batch_size= 1)

print(transcript)

['mimi anaitwa nathan una akiukweli napenda walisamaki yaani akiongelea walisamaki naongelea ni rosten ni ule wale ambao yaani samaki wake unakuwa unaoroja uroja yaani unakuwa mtaa tunaopenda']


In [9]:
def segment_audio(waveform, sr, chunk_duration= 5, overlap= 0.5):
    """
    Args:
        waveform: audio data in torch.Tensor format
        sr: sample rate i.e, number of audio samples in a second
        chunk_duration: how long a chunk is, defaults to 5 as defined in this function
        overlap: overlap ratio (0-1), eg. o.5 overlap means 50% overlap

    Returns:
        List of tuples, containing the chunk data, it's start time and end time
    """
    chunk_size = int(sr * chunk_duration)
    hop_size = int(chunk_size * (1 - overlap)) #stride between chunks

    chunks = []
    start_idx = 0

    while start_idx < len(waveform):
        end_idx = min(start_idx + chunk_size, len(waveform)) #for the last chunk, it's usually not the full chunk size
        chunk = waveform[start_idx:end_idx]

        start_time = start_idx / sr
        end_time = end_idx / sr

        chunks.append((chunk, start_time, end_time))
        start_idx += hop_size

    return chunks

In [10]:
chunks = segment_audio(waveform, sr)
chunks[-1]

(tensor([[0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00],
         ...,
         [0.0000e+00, 0.0000e+00],
         [6.1035e-05, 6.1035e-05],
         [6.1035e-05, 6.1035e-05]]),
 17.5,
 17.536)

In [11]:
transcripts = []
for chunk, start_time, end_time in chunks:
    with tempfile.NamedTemporaryFile(suffix= '.wav', delete= False) as tmp:
        sf.write(tmp.name, chunk.numpy(), sr)
        transcript = pipeline.transcribe([tmp.name], batch_size= 1, lang= ['swh_Latn'])[0]
        transcripts.append({
            'text': transcript,
            'start': start_time,
            'end': end_time,
        })
        print(f"{start_time:.1f}s - {end_time:.1f}s: {transcript}")

0.0s - 5.0s: mimi anaitwa nasa na kiukweli napenda wali sanaki
2.5s - 7.5s: ukweli napenda walisamaki yaani nikiongelea walisamaki
5.0s - 10.0s: ni kiongelea walisamaki naongelea ni rosteni
7.5s - 12.5s: naongelea ni rosten ni ule wale ambao yanisema kila
10.0s - 15.0s: ni ule wale ambao yani samaki wake unakuwa uneno unaroja uroja yaani unakuwa
12.5s - 17.5s: lakini unakuwa unaenda unaroja uroja yaani unakuwa mtaa tunaopenda
15.0s - 17.5s: tamu tunaopenda
17.5s - 17.5s: unini


In [12]:
[item['text'] for item in transcripts]

['mimi anaitwa nasa na kiukweli napenda wali sanaki',
 'ukweli napenda walisamaki yaani nikiongelea walisamaki',
 'ni kiongelea walisamaki naongelea ni rosteni',
 'naongelea ni rosten ni ule wale ambao yanisema kila',
 'ni ule wale ambao yani samaki wake unakuwa uneno unaroja uroja yaani unakuwa',
 'lakini unakuwa unaenda unaroja uroja yaani unakuwa mtaa tunaopenda',
 'tamu tunaopenda',
 'unini']

In [13]:
def merge_transcripts(transcripts):
    if not transcripts:
        return ""
    merged = transcripts[0]['text']

    for i in range(1, len(transcripts)):
        prev_end = transcripts[i-1]['end']
        curr_start = transcripts[i]['start']
        curr_text = transcripts[i]['text']

        #Add non overlapping part if there's overlap
        if curr_start < prev_end:
            #The overlap divided by total length of segment
            overlap_ratio = (prev_end - curr_start) / (transcripts[i]['end'] - curr_start)
            words = curr_text.split()
            skip_words = int(len(words) * overlap_ratio)
            non_overlapping = " ".join(words[skip_words:])
            merged += " "+ non_overlapping

        else:
            merged += " "+ curr_text
    return merged

In [14]:
all_transcripts= merge_transcripts(transcripts)
all_transcripts

'mimi anaitwa nasa na kiukweli napenda wali sanaki yaani nikiongelea walisamaki naongelea ni rosteni ule wale ambao yanisema kila wake unakuwa uneno unaroja uroja yaani unakuwa uroja yaani unakuwa mtaa tunaopenda tunaopenda '